In [ ]:
!pip -q install tensorflow

In [ ]:
# BETTER TO TRAIN WITH TPU v5e-1

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from tqdm import tqdm
import joblib
import json
import glob
import pickle
import sys
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # to avoid warnings

from google.colab import files
from zipfile import ZipFile

import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Layer, BatchNormalization, Dropout, LayerNormalization
from tensorflow.keras.activations import relu, tanh, sigmoid, swish, elu
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ModelCheckpoint
#from sklearn.model_selection import train_test_split  # not used
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import  StandardScaler


os.makedirs('resultados', exist_ok=True)  # to save all results

In [ ]:
# MAIN GENERAL VARIABLES
#####################################################################################################################
epocas = 400                 # maximum training epochs
batch = 8                    # batch size
val_split = 0.2              # split percentage of training data for validation

early_stop_patience = 60     # epochs for training early stopping criteria
early_stop_min_delta = 0.001  # considers 'no improvement' if improvement is < min_delta, to stop training
start_earlystop = 10         # waiting epochs before counting (stopping criteria)

reduce_lr_factor = 0.7       # learning rate reduction factor
reduce_lr_patience = 10       # epoch count to measure improvement
reduce_lr_min_delta = 0.0015   # considers 'no improvement' if improvement is < min_delta, to reduce learning rate
lr_min = 1e-9                # minimum learning rate
initial_lr = 3e-4            # initial learning rate
cool_down = 2                # waiting epochs before counting after lr reduction

#Loss = 'mean_squared_error'  # metric for loss
Loss = tf.keras.losses.Huber(delta=0.015)
MAE = 'mae'                  # auxiliary metric in training

dpi = 300                    # for saving images
IQR_factor = 1.5             # for outlier detection (pre-processing)
#####################################################################################################################

**SUBIR ARCHIVOS**

In [ ]:
# UPLOAD THE FF.py MODULE

files.upload()

from FF import crear_modelo, FFW_Act, FFW_L2, FFW_Layers

In [ ]:
# UPLOAD REQUIRED FILES

print("Upload files: Espectros.zip, wavelengths.csv, and temperature_values.csv")
uploaded = files.upload()


# extract zip containing the spectra
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with ZipFile(fname, 'r') as zip_ref:
            zip_ref.extractall()
        print(f"Extracted: {fname}")

In [ ]:
# DEFINE PATHS, CROP RANGE, AND SPECTRUM COUNT SPLIT FOR TRAIN/TEST

ruta = './Espectros_rename'
ruta_espectros = ruta  # spectrum CSVs are inside this folder
ruta_waves = './wavelengths.csv'
ruta_tabla = './temperature_values.csv'


# λ range
λ_min = float(input("Enter the minimum wavelength value in [nm]: "))
λ_max = float(input("Enter the maximum wavelength value in [nm]: "))

print(f"Selected wavelength range: {λ_min} nm, {λ_max} nm")
print(f"Files ready to process in: {ruta_espectros}")


#------------------------------------------------------------------------------------------------#
#------------------------------------------------------------------------------------------------#
# number of spectra for train + test (in order)
num_train_test = int(input("Enter the spectrum number from which Train + Test are split. Starting from 1: "))

# number of spectra for train + val (in order)
num_train_val = int(input("Enter the spectrum number where Val starts within Train: "))

**SPECTRA CROP AND CLEANING**



In [ ]:
# read: wavelengths
wave_df = pd.read_csv(ruta_waves, header=None)
wavelengths = wave_df.iloc[:,0].values

# read: temperatures
tabla_df     = pd.read_csv(ruta_tabla, header=None)
temperaturas = np.round(tabla_df.iloc[:, 0].values, 4)


#------------------------------------------------------------------------------------------------#
# crop spectra
espectros = []
csv_files = glob.glob(os.path.join(ruta_espectros, '*.csv'))

for i in tqdm(range(len(temperaturas)), desc="Processing spectra...", ncols=100, leave=False):

    inten = pd.read_csv(os.path.join(ruta, f'{i+1}.csv'), # read intensities
                        header=None).iloc[:,0].values

    # two-dimensional array [λ, I]
    data = np.vstack((wavelengths, inten)).T

    mask = (data[:,0] >= λ_min) & (data[:,0] <= λ_max)  # crop λ_min - λ_max
    espectros.append(data[mask])

X = np.array(espectros)    # shape: (M, Ni, 2)
y = np.array(temperaturas) # shape: (M,)

#------------------------------------------------------------------------------------------------#
print(f"Processed spectra count:  {len(X)}")
print(f"Loaded temperatures count:  {len(y)}")

In [ ]:
# Verify crop

# Show a random spectrum
idx = np.random.randint(1, len(X) + 1)

plt.figure(figsize=(8,4))
plt.plot(X[idx,:,0], X[idx,:,1], label=f'Spectrum {idx}')
plt.xlabel('λ [nm]')
plt.title(f'Cropped spectrum {idx} ({λ_min}–{λ_max} nm)')
plt.show()

In [ ]:
# remove NaN data (if any)

print("Original data count:", X.shape[0])
#------------------------------------------------------------------------------------------------#

mask_nan = ~np.isnan(X).any(axis=(1, 2))  # 3D X
X, y = X[mask_nan], y[mask_nan]
print(f"After removing NaNs: {X.shape[0]} samples (discarded {np.sum(~mask_nan)})")

#------------------------------------------------------------------------------------------------#
# remove outliers in y (if any, using IQR criterion)

Q1, Q3 = np.percentile(y, [25, 75]) # calculate IQR to remove outliers
IQR = Q3 - Q1
lower_bound = Q1 - IQR_factor * IQR # see whether to keep this 1.5 factor or change it!!
upper_bound = Q3 + IQR_factor * IQR

mask_outliers = (y >= lower_bound) & (y <= upper_bound)
print(f"Outliers detected and discarded: {np.sum(~mask_outliers)}")

X, y = X[mask_outliers], y[mask_outliers]
print(f"After removing outliers: {X.shape[0]} samples")

#------------------------------------------------------------------------------------------------#
# verify final dimensions
print("Dimension of X: ", X.shape)
print("Dimension of y: ", y.shape)

**DATA SPLITTING AND SCALING**

In [ ]:
# Split for training:

#------------------------------------------------------------------------------------------------#
# MANUAL SPLIT (TRAIN AND TEST):
X_train, X_test = X[:num_train_test], X[num_train_test:]
y_train, y_test = y[:num_train_test], y[num_train_test:]

# MANUAL SPLIT (TRAIN AND VAL):
X_train, X_val = X_train[:num_train_val], X_train[num_train_val:]
y_train, y_val = y_train[:num_train_val], y_train[num_train_val:]
#------------------------------------------------------------------------------------------------#


# Flatten X if it comes in 3D (e.g., (n_samples, 104, 2))
X_train = X_train.reshape(X_train.shape[0], -1)  # (n_samples, 208)
X_test  = X_test.reshape(X_test.shape[0], -1)
X_val   = X_val.reshape(X_val.shape[0], -1)

# Scaling:
#------------------------------------------------------------------------------------------------#
# Label scaling (T)
scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_val   = scaler_y.transform(y_val.reshape(-1, 1))
y_test  = scaler_y.transform(y_test.reshape(-1, 1))

# Intensity scaling (X)
scaler_x = StandardScaler()
X_train = scaler_x.fit_transform(X_train)  # already in 2D
X_val   = scaler_x.transform(X_val)
X_test  = scaler_x.transform(X_test)


scale_T = float(scaler_y.scale_[0])  # std of T (°C) - to monitor training in °C


# tensor shapes
#------------------------------------------------------------------------------------------------#
print("X_train shape: ", X_train.shape)
print("X_val shape: ", X_val.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape: ", y_train.shape)
print("y_val shape: ", y_val.shape)
print("y_test shape: ", y_test.shape)

#------------------------------------------------------------------------------------------------#
print('Points per spectrum:', X_train.shape[1])

**MODEL COMPILATION**



In [ ]:
# COMPILE THE MODEL:
#------------------------------------------------------------------------------------------------#
input_shape = (X_train.shape[1],)  # in feed-forward
model = crear_modelo(input_shape)
#------------------------------------------------------------------------------------------------#

#------------------------------------------------------------------------------------------------#
# metrics to monitor training in °C (training still uses scaled data)
def mse_C(y_true, y_pred):
    return tf.sqrt( (scale_T**2) * tf.reduce_mean(tf.square(y_true - y_pred)) )
def mae_C(y_true, y_pred):
    return scale_T * tf.reduce_mean(tf.abs(y_true - y_pred))
#------------------------------------------------------------------------------------------------#


# compile model
model.compile(optimizer=Adam(learning_rate= initial_lr), loss= Loss, metrics=[MAE, mse_C, mae_C], # ADD METRICS HERE
              jit_compile= False, run_eagerly= False)

#------------------------------------------------------------------------------------------------#
model.summary()

**TRAINING**

In [ ]:
# callbacks
#------------------------------------------------------------------------------------------------#
# checkpoint
ckpt = ModelCheckpoint(filepath="resultados/best_val_loss.weights.h5", monitor="val_mae_c", mode="min",  # val_loss
    save_best_only=True, save_weights_only=True, verbose=0 )

# early stopping
early_stop = EarlyStopping(monitor='val_mae_c', patience= early_stop_patience, min_delta=1e-9,  # val_loss
    restore_best_weights=True, verbose=1, mode= 'min' )

# reduce learning rate during training
reduce_lr = ReduceLROnPlateau(monitor='val_mae_c', factor= reduce_lr_factor,                    # val_loss
    patience= reduce_lr_patience, min_delta= reduce_lr_min_delta, min_lr= lr_min,
    restore_best_weights=True, cooldown= cool_down, verbose=1, mode= 'min' )
#------------------------------------------------------------------------------------------------#


# Training begins
#------------------------------------------------------------------------------------------------#
history = model.fit(X_train, y_train, epochs= epocas, batch_size= batch,
    validation_data= (X_val, y_val), callbacks=[ckpt, early_stop, reduce_lr])


df_hist = pd.DataFrame(history.history)

#------------------------------------------------------------------------------------------------#
output_file = os.path.join('resultados/training.csv')
df_hist.to_csv(output_file, index=False)

**VISUALIZATION OF RESULTS**

In [ ]:
# Training Loss vs Validation Loss
#------------------------------------------------------------------------------------------------#
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylim( 0,  max(history.history['val_loss']) )
plt.legend()

plt.savefig(os.path.join("resultados/val_loss.png"), dpi= dpi, bbox_inches='tight')
plt.show()



# Training MAE vs Validation MAE in °C
#------------------------------------------------------------------------------------------------#
plt.plot(history.history['mae_c'], label='Training MAE (°C)')
plt.plot(history.history['val_mae_c'], label='Validation MAE (°C)')
plt.xlabel('Epochs')
plt.ylim(0, max(history.history['val_mae_c']))
plt.legend()

plt.savefig(os.path.join("resultados/val_mae_c.png"), dpi=dpi, bbox_inches='tight')
plt.show()

In [ ]:
# TEST PREDICTIONS - FEEDFORWARD

y_pred_scaled = model.predict(X_test)  # Predict (scaled)
#------------------------------------------------------------------------------------------------#

# inverse scaling
y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
y_test = scaler_y.inverse_transform(y_test).flatten()


# save predictions
df_preds = pd.DataFrame({'y_pred': y_pred, 'y_test': y_test})
df_preds.to_csv(os.path.join("resultados/predicciones.csv"), index=False)


#------------------------------------------------------------------------------------------------#
print(f"y_pred shape: {y_pred.shape}")
print(f"y_test shape: {y_test.shape}")
print("X_test shape:", X_test.shape)

In [ ]:
# post-hoc linear calibration (val -> adjusts test) --- optional but very useful
#------------------------------------------------------------------------------------------------#

# val in °C
y_val_pred_scaled = model.predict(X_val)
y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1,1)).flatten()
y_val_true = scaler_y.inverse_transform(y_val.reshape(-1,1)).flatten()

#------------------------------------------------------------------------------------------------#
# linear fit on validation
a, b = np.polyfit(y_val_pred, y_val_true, 1)

# apply to TEST (overwrites y_pred already in °C)
y_pred = a*y_pred + b
#------------------------------------------------------------------------------------------------#


#------------------------------------------------------------------------------------------------#
# re-save predictions (so your metrics block reads them already calibrated)
df_preds = pd.DataFrame({'y_pred': y_pred, 'y_test': y_test})
df_preds.to_csv(os.path.join("resultados/predicciones_cal.csv"), index=False)

In [ ]:
# Prediction metrics on the real scale (°C) – Test

df = pd.read_csv('resultados/predicciones.csv')
df_cal = pd.read_csv('resultados/predicciones_cal.csv')

pred = df['y_pred']
pred_cal = df_cal['y_pred']

test = df['y_test']

mse = mean_squared_error(test, pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test, pred)

mse_cal = mean_squared_error(test, pred_cal)
rmse_cal = np.sqrt(mse_cal)
mae_cal = mean_absolute_error(test, pred_cal)

print(f"RMSE: {np.round(rmse, 5)}")
print(f"MAE: {np.round(mae, 5)}")

print(f"RMSE_cal: {np.round(rmse_cal, 5)}")
print(f"MAE_cal: {np.round(mae_cal, 5)}")


#------------------------------------------------------------------------------------------------#
with open('resultados/metricas_°C.txt', 'w') as f:
    f.write("RMSE\tMAE\n")
    f.write(f"{np.round(rmse, 7)}\t{np.round(mae, 7)}\n")

with open('resultados/metricas_°C_cal.txt', 'w') as f:
    f.write("RMSE\tMAE\n")
    f.write(f"{np.round(rmse, 7)}\t{np.round(mae, 7)}\n")

In [ ]:
# graphs - predictions:

#------------------------------------------------------------------------------------------------#
# pred vs test
plt.scatter(y_test, y_pred,  s=2)
plt.xlabel('y_test (° C)')
plt.ylabel('y_pred (° C)')
plt.title('Predictions')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red')  # reference line
plt.savefig(os.path.join('resultados/Predictions.png'), format='png', dpi= dpi, bbox_inches='tight')
plt.show()

#------------------------------------------------------------------------------------------------#
residuos = y_test - y_pred  # residuals

# residuals plot
plt.scatter(y_test, residuos, s=2)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('y_test  (° C)')
plt.ylabel('Residuals = y_test - y_pred')
plt.ylim(min(residuos), max(residuos))
plt.savefig(os.path.join('resultados/Residuals.png'), format='png', dpi= dpi, bbox_inches='tight')
plt.show()

#------------------------------------------------------------------------------------------------#
# residuals fft
residuos_fft = fft(residuos)
frecuencias = fftfreq(len(residuos))

plt.plot(frecuencias[:len(frecuencias)//2], np.abs(residuos_fft[:len(residuos_fft)//2]))
plt.xlabel('Frequency')
plt.title('Residuals FFT')
plt.show()

In [ ]:
# Predictions in the Test cycle (temperature curve)

temps = pd.read_csv(os.path.join('temperature_values.csv')).iloc[:, 0].values

true_test = temps[num_train_test:]
pred_test = y_pred

# adjust to the same length if there is a mismatch
n = min(len(true_test), len(pred_test))
true_test = true_test[:n]
pred_test = pred_test[:n]

x = range(num_train_test, num_train_test + n) # index (spectrum number)


#---------------------------------------------------------------------------------------------
# graph
plt.figure(figsize=(10, 4))
plt.plot(x, true_test, linestyle='-', linewidth=1.5, label='Measured temperature')
plt.scatter(x, pred_test,  color='red', marker='o', s=0.3,   label='Predicted temperature')
plt.xlabel('Spectrum number')
plt.ylabel('Temperature (°C)')
plt.legend()

# save predictions (°C) in test
f_out = pd.DataFrame({'indice': x, 'temp_real': true_test, 'temp_predicha': pred_test })
f_out.to_csv(os.path.join('resultados/T_vs_Pred.csv'), index=False)


plt.savefig(os.path.join('resultados/T_vs_Pred.png'), dpi= dpi)
plt.show()


In [ ]:
# Distribution of y_pred-y_test (histogram)

df = pd.read_csv('resultados/predicciones.csv')

df['diff'] = (df['y_pred'] - df['y_test'])

std_diff = df['diff'].std()
print(f'Standard deviation of differences: {std_diff:.4f}')


#---------------------------------------------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.hist(df['diff'], bins=30)
plt.xlabel('y_pred - y_test (°C)')
plt.ylabel('Frequency')

mean_diff = df['diff'].mean()

plt.axvline(mean_diff, color='red', linestyle='--', linewidth=1.5)
plt.axvline(0, color='green', linestyle='--', linewidth=1.5)

x_max, y_max = plt.xlim()[1], plt.ylim()[1]
texto = f"Mean: {mean_diff:.3f} °C\nStd: {std_diff:.3f} °C"
plt.text(x_max-0.2, y_max-10, texto, ha='right', va='top', fontsize=11,
         bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

plt.savefig("resultados/err_dist.png", dpi= dpi)

plt.show()

In [ ]:
#---------------------------------------------------------------------------------------------
# Offset correction on the test set
#---------------------------------------------------------------------------------------------


# Offset in °C
offset = df['diff'].mean()
print(f"Offset (mean of y_pred - y_test): {offset:.5f} °C")


#---------------------------------------------------------------------------------------------
# Corrected predictions (removing the offset)
df['y_pred_corr'] = df['y_pred'] - offset
df['diff_corr'] = df['y_pred_corr'] - df['y_test']  # New error: y_pred_corr - y_test

mean_diff_corr = df['diff_corr'].mean()
std_diff_corr  = df['diff_corr'].std()
rmse_corr = np.sqrt(mean_squared_error(df['y_test'], df['y_pred_corr'])) # RMSE in corrected predictions
mae_corr  = mean_absolute_error(df['y_test'], df['y_pred_corr'])         # MAE in corrected predictions


#---------------------------------------------------------------------------------------------
print(f"Mean after correcting offset: {mean_diff_corr:.5f} °C")
print(f"Std after correcting offset:   {std_diff_corr:.5f} °C")
print(f"Corrected RMSE: {rmse_corr:.5f} °C")
print(f"Corrected MAE: {mae_corr:.5f} °C")

In [ ]:
# Save new predictions and metrics with corrected offset

df.to_csv("resultados/predicciones_corr.csv", index=False)

with open('resultados/metricas_escala_real-°C_calibradas.txt', 'w') as f:
    f.write("offset(°C)\tRMSE_corr\tMAE_corr\tmean_diff_corr\tstd_diff_corr\n")
    f.write(f"{offset:.7f}\t{rmse_corr:.7f}\t{mae_corr:.7f}\t"
            f"{mean_diff_corr:.7f}\t{std_diff_corr:.7f}\n")

In [ ]:
# Histogram (y_pred-y_test) after offset correction

plt.figure(figsize=(8, 5))
plt.hist(df['diff_corr'], bins=30)
plt.xlabel('y_pred_corr - y_test (°C)')
plt.ylabel('Frequency')

plt.axvline(0, color='green', linestyle='--', linewidth=2)

x_max, y_max = plt.xlim()[1], plt.ylim()[1]
texto = f"Mean: {mean_diff_corr:.3f} °C\nStd: {std_diff_corr:.3f} °C"
plt.text(x_max - 0.2, y_max - 10, texto, ha='right', va='top', fontsize=11,
         bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))

plt.savefig("resultados/err_dist_cal_corr.png", dpi= dpi)
plt.show()

In [ ]:
# Measured vs Predicted T curve - OFFSET CORRECTED - test cycle

n = len(df)  # number of test samples

true_test = df['y_test'].values
pred_test_corr = df['y_pred_corr'].values

x = np.arange(n)


#---------------------------------------------------------------------------------------------
plt.figure(figsize=(10, 4))
plt.plot(x, true_test, linestyle='-', linewidth=1.5, label='Measured temperature')
plt.scatter(x, pred_test_corr, color='red', marker='o', s=0.3, label='Predicted temperature (corr.)')
plt.xlabel('Spectrum index')
plt.ylabel('Temperature (°C)')
plt.legend()

f_out_corr = pd.DataFrame({'indice': x, 'temp_real': true_test, 'temp_pred_corrida': pred_test_corr})
f_out_corr.to_csv(os.path.join('resultados/T_vs_Pred_cal_corr.csv'), index=False)

plt.savefig(os.path.join('resultados/T_vs_Pred_cal_corr.png'), dpi= dpi)
plt.show()

**SAVING**

In [ ]:
# save model, scalers, and metadata

#------------------------------------------------------------------------------------------------#
model_path = os.path.join(f"resultados/FFW-trained{epocas}ep_{λ_min}-{λ_max}nm.h5")
model.save(model_path)

#------------------------------------------------------------------------------------------------#
# save scalers
joblib.dump(scaler_x, "resultados/scaler_x.pkl")  # intensities
joblib.dump(scaler_y, "resultados/scaler_y.pkl")  # temperatures


#------------------------------------------------------------------------------------------------#
# save useful data
meta = {
    "λ_min": float(λ_min),
    "λ_max": float(λ_max),
    "offset_T":   float(offset),   # in °C

    "model": {
        "type": "FFW",
        "activation": str(FFW_Act),
        "l2": float(FFW_L2),
        "layers": [int(u) for u in FFW_Layers] }   }

    # add any other relevant data


with open("resultados/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)


**DOWNLOAD RESULTS AND CLEAR SESSION**


In [ ]:
# DOWNLOAD RESULTS
!zip -r Resuls.zip resultados

files.download('Results.zip')

In [ ]:
import shutil
# CLEAN UP COLAB SESSION

# Deletes all folders and files (except the notebook)
for nombre in os.listdir():
    if nombre != 'sample_data':  # to avoid touching the default Colab folder
        try:
            ruta = os.path.join(os.getcwd(), nombre)
            if os.path.isfile(ruta):
                os.remove(ruta)
            elif os.path.isdir(ruta):
                shutil.rmtree(ruta)
        except Exception as e:
            print(f"Error deleting {nombre}: {e}")

print("Files and folders deleted from the session.")